# 05 — Baseline Sentiment Classification

Train comparable TF-IDF baselines using Logistic Regression, Naive Bayes and Linear SVM. Use validation data for model selection and reserve the test set for final evaluation.

## 1. Load and validate the cleaned dataset

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

cleaned_data = pd.read_csv("../data/processed/montra_cleaned_reviews.csv")
LABELS = ["negative", "neutral", "positive"]

cleaned_data["sentiment_label"] = (
    cleaned_data["sentiment_label"].astype(str).str.lower().str.strip()
)
cleaned_data = cleaned_data[
    cleaned_data["review_text_clean"].notna()
    & cleaned_data["review_text_clean"].astype(str).str.strip().ne("")
    & cleaned_data["sentiment_label"].isin(LABELS)
].copy()

print(cleaned_data["sentiment_label"].value_counts())

## 2. Create train, validation and test splits

In [ ]:
from sklearn.model_selection import train_test_split

# 60% train, 20% validation, 20% test.
train_df, temp_df = train_test_split(
    cleaned_data,
    test_size=0.40,
    random_state=42,
    stratify=cleaned_data["sentiment_label"]
)
valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["sentiment_label"]
)

X_train, y_train = train_df["review_text_clean"], train_df["sentiment_label"]
X_valid, y_valid = valid_df["review_text_clean"], valid_df["sentiment_label"]
X_test, y_test = test_df["review_text_clean"], test_df["sentiment_label"]

print(len(train_df), len(valid_df), len(test_df))

## 3. Train and compare baseline models

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt
import joblib

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(class_weight="balanced")
}

fitted_models = {}
validation_results = []

for name, classifier in models.items():
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(max_features=20000, ngram_range=(1, 2))),
        ("model", classifier)
    ])
    pipeline.fit(X_train, y_train)
    pred = pipeline.predict(X_valid)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_valid, pred, labels=LABELS, average="macro", zero_division=0
    )
    validation_results.append({
        "model": name,
        "accuracy": accuracy_score(y_valid, pred),
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1
    })
    fitted_models[name] = pipeline

validation_results = pd.DataFrame(validation_results).sort_values(
    "macro_f1", ascending=False
)
display(validation_results)

## 4. Select model using validation macro-F1

In [ ]:
best_name = validation_results.iloc[0]["model"]
best_model = fitted_models[best_name]
print("Selected baseline:", best_name)

# Evaluate the selected baseline once on the held-out test set.
test_pred = best_model.predict(X_test)
print(classification_report(
    y_test, test_pred, labels=LABELS, zero_division=0
))

ConfusionMatrixDisplay.from_predictions(
    y_test, test_pred, labels=LABELS, xticks_rotation=45
)
plt.title(f"Confusion Matrix — {best_name}")
plt.tight_layout()
plt.show()

## 5. Save model and test predictions

In [ ]:
OUT = Path("../models/baseline")
OUT.mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, OUT / "best_baseline_pipeline.joblib")

predictions = test_df.copy()
predictions["predicted_sentiment"] = test_pred
predictions.to_csv("../data/processed/baseline_test_predictions.csv", index=False)
validation_results.to_csv("../reports_baseline_validation.csv", index=False)
print("Saved baseline model and predictions.")